# Exercise: Missing Values

In [64]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Read data
X_full = pd.read_csv('../../data/home_data/train.csv', index_col='Id') # ../ goes up one directory then into the next folders
X_test_full = pd.read_csv('../../data/home_data/test.csv', index_col='Id') # index_col is column used as index

# Remove rows with missing target, separate target from predictors
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True) # subset parameter is list of columns to consider checking for missing values, inplace True modifies the dataframe instead of creating a new one
y = X_full.SalePrice
X_full.drop(['SalePrice'], axis=1, inplace=True) # drop method removes specified rows/columns

# To keep things simple we'll use only numerical predictors
X = X_full.select_dtypes(exclude=['object']) # exclude categorical variables (object)
X_test = X_test_full.select_dtypes(exclude=['object'])

# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

In [65]:
# Print first 5 rows
X_train.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
Id,,,,,,,,,,,,,,,,,,,,,
619,20,90.0,11694,9,5,2007,2007,452.0,48,0,...,774,0,108,0,0,260,0,0,7,2007
871,20,60.0,6600,5,5,1962,1962,0.0,0,0,...,308,0,0,0,0,0,0,0,8,2009
93,30,80.0,13360,5,7,1921,2006,0.0,713,0,...,432,0,0,44,0,0,0,0,8,2009
818,20,NaN,13265,8,5,2002,2002,148.0,1218,0,...,857,150,59,0,0,0,0,0,7,2008
303,20,118.0,13704,7,5,2001,2002,150.0,0,0,...,843,468,81,0,0,0,0,0,1,2006


In [66]:
# Shape of training data (num_rows, nul_columns)
print(X_train.shape)

# Number of missing values in each column of training data
missing_val_count_by_column = (X_train.isnull().sum()) # sum of true isnull values in each column
print(missing_val_count_by_column[missing_val_count_by_column > 0]) # hide columns with no missing values

(1168, 36)
LotFrontage    212
MasVnrArea       6
GarageYrBlt     58
dtype: int64


total rows: 1168

columns (features) contain missing values: 3

total missing entries: 212 + 6 + 58 = 276

In [67]:
from sklearn.ensemble import RandomForestRegressor

# Define model
model = RandomForestRegressor(n_estimators=100, random_state=0) # n_estimators is number of trees in forest

names of columns with missing values: LotFrontage, MasVnrArea, GarageYrBlt

In [68]:
cols_with_missing = [ # list of columns with missing values
    col for col in X_train.columns # append col in list
    if X_train[col].isnull().any() # if there is at least one missing value
]

# Drop columns in training and validation data
reduced_X_train = X_train.drop(cols_with_missing, axis=1) # axis=1 is columns
reduced_X_valid = X_valid.drop(cols_with_missing, axis=1)

In [69]:
import sys
import os

sys.path.append(os.path.abspath('../..')) # add parent directory to search path for modules

from src.utilities import score_model

print("MAE (Drop columns with missing values):")
print(score_model(model, reduced_X_train, reduced_X_valid, y_train, y_valid))

MAE (Drop columns with missing values):
17837.82570776256


In [70]:
from sklearn.impute import SimpleImputer

my_imputer = SimpleImputer() # by default, imputes mean
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train)) # fit imputer to training data
imputed_X_valid = pd.DataFrame(my_imputer.transform(X_valid)) # apply imputer to validation data, don't fit

# Imputation removes column names; put them back
imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns

In [71]:
print("MAE (Imputation):")
print(score_model(model, imputed_X_train, imputed_X_valid, y_train, y_valid))

MAE (Imputation):
18062.894611872147


Given that there are so few missing values in the dataset, imputation was expected to yield better results. However, dropping results instead had a lower MAE. While noise in the dataset can partially be attributed to the results, another potential explanation is that imputation wasn't the best method for this dataset. 
Maybe instead of filling in the null values with the mean, it might make more sense to fill in values with 0, to fill in with values most frequently encountered (mode), or some other method of getting a value for imputing. For instance, the GarageYrBlt column, it's most likely that in some cases, a missing value indicates a house that doesn't have a garage. That begs the question, what value would be the best to fill in this case? Would it make more sense to fill in the median value? Or maybe with the minimum value? There are some options we can rule out immediately; setting missing values in this column to 0 is likely to yield terrible results.

In [72]:
median_imputer = SimpleImputer(strategy='median') # impute with median
mode_imputer = SimpleImputer(strategy='most_frequent') # impute with mode (most common values)

median_X_train = pd.DataFrame(median_imputer.fit_transform(X_train)) # fit imputer to training data
median_X_valid = pd.DataFrame(median_imputer.transform(X_valid)) # impute median on validation data

mode_X_train = pd.DataFrame(mode_imputer.fit_transform(X_train)) # fit imputer to training data
mode_X_valid = pd.DataFrame(mode_imputer.transform(X_valid)) # impute mode on validation data

# Imputation removes column names; put them back
median_X_train.columns = X_train.columns
median_X_valid.columns = X_valid.columns

mode_X_train.columns = X_train.columns
mode_X_valid.columns = X_valid.columns

we won't be using the score_model() function in utilities as we will be using the trained model to generate test predictions for 'X_test' in first code cell

In [73]:
from sklearn.metrics import mean_absolute_error

# RandomForestRegressor already defined in previous cell as 'model'

# Fit, predict, and score both imputation methods
model.fit(median_X_train, y_train)
median_preds_valid = model.predict(median_X_valid)
print("MAE (Median imputation):")
print(mean_absolute_error(y_valid, median_preds_valid))

model.fit(mode_X_train, y_train)
mode_preds_valid = model.predict(mode_X_valid)
print("MAE (Mode imputation):")
print(mean_absolute_error(y_valid, mode_preds_valid))

MAE (Median imputation):
17791.59899543379
MAE (Mode imputation):
17956.065479452056


since imputation with the median value seems to yield more accurate results (lower MAE), we will be using it as that as the preferred method to use on X_test

In [74]:
final_imputer = median_imputer
final_X_train = median_X_train

model.fit(final_X_train, y_train)

# Preprocess test data
final_X_test = pd.DataFrame(final_imputer.transform(X_test)) # apply imputer to test data
final_X_test.columns = X_test.columns # imputation removed column names

# Get test predictions
preds_test = model.predict(final_X_test)

create a csv file with id and sales predictions below

In [75]:
# create output dataframe with Id and SalePrice (predictions)
output = pd.DataFrame(
    {
        'Id': X_test.index, # create Id column, X_test.index are the house indices in X_test
        'PredictedSalePrice': preds_test # create PredictedSalePrice column, preds_test are the predicted prices
    }
)

# uncomment the line below to create a csv file
# output.to_csv('submission.csv', index=False) # create csv from 'output' dataframe, index=False removes index column